# Decoupled Simulation Demo

This notebook demonstrates the new decoupled pipeline:
1. Run expensive pyFAI synthesis once to create a clean base pattern.
2. Store the base pattern in HDF5.
3. Apply domain randomization dynamically when loading samples.

In [ ]:
from pathlib import Path
import numpy as np
import h5py
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from pyFAI.calibrant import CALIBRANT_FACTORY
from pyFAI.detectors import detector_factory

from src.data_pipeline import CalibrantSimulator, DiffractionDataset
from src.detectors import DetectorProfile

In [ ]:
wavelength = 0.5121261413149675e-10
calibrant_name = "alpha_Al2O3"
detector_name = "Eiger2Cdte_1M"

geometry = {
    "dist": 0.11,
    "poni1": 0.0425,
    "poni2": 0.0375,
    "rot1": 0.01,
    "rot2": -0.01,
    "rot3": 0.005,
}

calibrant = CALIBRANT_FACTORY(calibrant_name)
calibrant.wavelength = wavelength
detector = detector_factory(detector_name)

sim = CalibrantSimulator(calibrant, detector, geometry, wavelength)

base_image = sim.run(
    imax=150.0,
    imin=4.0,
    fwhm=0.10,
    k_alpha_ratio=0.5,
    apply_domain_randomization=False,
    randomize_occlusions=False,
)

print(f"Base image shape: {base_image.shape}, dtype: {base_image.dtype}")

In [ ]:
demo_dir = Path("../datasets")
demo_dir.mkdir(parents=True, exist_ok=True)
demo_h5 = demo_dir / "decoupled_demo.hdf5"

num_samples = 8
images = np.repeat(base_image[None, ...], num_samples, axis=0).astype(np.float32)
labels = np.tile(np.array(list(geometry.values()), dtype=np.float32), (num_samples, 1))

with h5py.File(demo_h5, "w") as hf:
    grp = hf.create_group("demo")
    grp.create_dataset("images", data=images, dtype="f4")
    grp.create_dataset("labels", data=labels, dtype="f4")

print(f"Wrote: {demo_h5}")

In [ ]:
profile = DetectorProfile(
    imin_range=(0.0, 0.5), 
    beamstop_prob=1.0,
    deadzone_prob=0.0,
    deadzone_count_range=(1, 3),
    deadzone_width_range=(2, 8),
    flux_range=(1.0, 5.0),                 
    scatter_intensity_range=(0.0, 10.0), 
    scatter_decay_range=(3e-5, 5e-5),      
    texture_floor_range=(0.55, 0.75),
    readout_noise_std_range=(0.0, 0.0),
)

ds_clean = DiffractionDataset(
    hdf5_path=str(demo_h5),
    group="demo",
    image_size=base_image.shape[0],
    apply_dynamic_randomization=False,
    detector_profile=profile,
)

ds_rand = DiffractionDataset(
    hdf5_path=str(demo_h5),
    group="demo",
    image_size=base_image.shape[0],
    apply_dynamic_randomization=True,
    detector_profile=profile,
)

clean_tensor, _ = ds_clean[0]
clean_img = clean_tensor[0].numpy()

rand_imgs = []
for _ in range(4):
    t, _ = ds_rand[0]
    rand_imgs.append(t[0].numpy())

print("Collected 1 clean + 4 randomized versions of the same base pattern.")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4), constrained_layout=True)

axes[0].imshow(clean_img, cmap="gray", origin="lower")
axes[0].set_title("Loaded clean")
axes[0].axis("off")

for i, img in enumerate(rand_imgs, start=1):
    axes[i].imshow(img, cmap="gray", origin="lower")
    axes[i].set_title(f"Randomized {i}")
    axes[i].axis("off")

plt.show()

In [ ]:
stack = np.stack(rand_imgs, axis=0)
diff_mean = np.mean(np.abs(stack - clean_img[None, ...]))
print(f"Mean |randomized - clean| over 4 loads: {diff_mean:.4f}")

In [ ]:
def _close_dataset_file(name: str):
    obj = globals().get(name)
    if obj is None:
        return
    
    file_attr = None
    obj_dict = getattr(obj, "__dict__", None)
    if isinstance(obj_dict, dict):
        file_attr = obj_dict.get("file")

    if file_attr is not None:
        try:
            file_attr.close()
        except Exception as exc:
            print(f"Could not close {name}.file: {exc}")
        finally:
            try:
                obj.file = None
            except Exception:
                pass
        print(f"Closed {name}.file")


def _close_handle(name: str):
    obj = globals().get(name)
    if obj is None:
        return

    close_fn = getattr(obj, "close", None)
    if callable(close_fn):
        try:
            close_fn()
            print(f"Closed {name}")
        except Exception as exc:
            print(f"Could not close {name}: {exc}")


for _name in ("ds_clean", "ds_rand"):
    _close_dataset_file(_name)

for _name in ("hf",):
    _close_handle(_name)

del _close_dataset_file
del _close_handle